In [1]:
import open3d as o3d
import numpy as np


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
np.random.seed(42)

mesh = o3d.geometry.TriangleMesh.create_sphere(
    radius=1.0,
    resolution=20
)

clean_vertices = np.asarray(mesh.vertices).copy()
triangles = np.asarray(mesh.triangles)

print("Vertices:", len(clean_vertices))
print("Triangles:", len(triangles))

Vertices: 762
Triangles: 1520


In [3]:
noise = np.random.normal(
    0, 0.05, clean_vertices.shape
)

noisy_vertices = clean_vertices + noise

In [4]:
noisy_mesh = o3d.geometry.TriangleMesh()

noisy_mesh.vertices = o3d.utility.Vector3dVector(noisy_vertices)
noisy_mesh.triangles = o3d.utility.Vector3iVector(triangles)

noisy_mesh.compute_vertex_normals()

TriangleMesh with 762 points and 1520 triangles.

In [5]:
noisy_mesh.compute_adjacency_list()

adjacency = noisy_mesh.adjacency_list

print("Vertex 0 neighbors:", adjacency[0])

Vertex 0 neighbors: {2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41}


In [6]:
LAMBDA = 0.5
ITERATIONS = 10

In [7]:
vertices = noisy_vertices.copy()

for _ in range(ITERATIONS):

    new_vertices = vertices.copy()

    for i, neighbors in enumerate(adjacency):

        if not neighbors:
            continue

        neighbor_avg = vertices[list(neighbors)].mean(axis=0)

        new_vertices[i] = (
            vertices[i]
            + LAMBDA * (neighbor_avg - vertices[i])
        )

    vertices = new_vertices

denoised_vertices = vertices

In [8]:
denoised_mesh = o3d.geometry.TriangleMesh()

denoised_mesh.vertices = o3d.utility.Vector3dVector(
    denoised_vertices
)

denoised_mesh.triangles = o3d.utility.Vector3iVector(
    triangles
)

denoised_mesh.compute_vertex_normals()

TriangleMesh with 762 points and 1520 triangles.

In [9]:
o3d.visualization.draw_plotly(
    [mesh]
)



In [10]:
o3d.visualization.draw_plotly(
    [noisy_mesh]
)


In [11]:
o3d.visualization.draw_plotly(
    [denoised_mesh]
)

In [12]:
noisy_error = np.linalg.norm(
    noisy_vertices - clean_vertices,
    axis=1
)

denoised_error = np.linalg.norm(
    denoised_vertices - clean_vertices,
    axis=1
)

print(f"Noisy mean error:     {noisy_error.mean():.6f}")
print(f"Denoised mean error:  {denoised_error.mean():.6f}")

print()

print(f"Noisy max error:      {noisy_error.max():.6f}")
print(f"Denoised max error:   {denoised_error.max():.6f}")

Noisy mean error:     0.078533
Denoised mean error:  0.071929

Noisy max error:      0.196989
Denoised max error:   0.100277


In [13]:
improvement = (
    1 - denoised_error.mean() / noisy_error.mean()
) * 100

print(f"Mean error improvement: {improvement:.2f}%")

Mean error improvement: 8.41%
